# 06. 외부데이터 전처리 → 격자 결합

UTCI(시간대별) 지수의 물리 입력 3종을 분석용으로 정제.

- **기상**(기상청 ASOS): 시간별 기온·습도·풍속 → `data/processed/weather_hourly.csv` (24h forcing)
- **건물**(VWorld, 실제높이): 격자별 건폐율·평균/최대 높이 → SVF·그림자 대리
- **DEM**(Copernicus 30m): 격자별 표고·TPI(지형위치) → 분지 열갇힘

건물·DEM은 `grid_features.geojson`에 컬럼 추가. 멱등.

In [1]:
# ── 실행 환경 가드 (하이브리드 전략, 실행순서.md §1 참조) ──────────────
# 이 노트북은 '로컬 사전계산' 단계입니다. COMPAS에는 ['rasterio'] 이(가) 없으므로
# 그 경우 아래 셀들은 자동으로 건너뛰고(LOCAL_RUN=False) 동봉 산출물만 검증합니다.
# 산출물은 제출 데이터 ZIP에 포함되어 있어 후속 노트북은 그대로 실행됩니다.
import importlib.util, pathlib
_DEPS = ['rasterio']
_missing = [m for m in _DEPS if importlib.util.find_spec(m) is None]
LOCAL_RUN = not _missing
_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
if LOCAL_RUN:
    print("[로컬 모드] 의존성 충족 - 전체 셀을 실행합니다.")
else:
    print("[COMPAS 모드] 미설치 의존성:", _missing, "-> 아래 셀은 건너뜁니다.")
    print("이 단계의 동봉 산출물 검증:")
    for _f in ['data/processed/weather_hourly.csv', 'data/processed/grid_features.geojson']:
        _p = _ROOT / _f
        if _p.exists():
            print("  [OK]", _f, "(%.1f MB)" % (_p.stat().st_size / 1e6))
        else:
            print("  [!!]", _f, "없음 - 제출 데이터 ZIP의 해당 파일을 이 경로에 두십시오.")


[로컬 모드] 의존성 충족 - 전체 셀을 실행합니다.


In [2]:
if LOCAL_RUN:
    import sys, pathlib
    sys.path.append(str(pathlib.Path.cwd().parent))
    import numpy as np, pandas as pd, geopandas as gpd
    from src.config import DATA_EXTERNAL, DATA_PROCESSED, OUT_FIGURES, CRS_LANDSAT, CRS_SUBMIT

    # 기상: 시간별 정제
    w = pd.read_csv(DATA_EXTERNAL / "weather" / "asos_hourly_2024summer.csv")
    w["tm"] = pd.to_datetime(w["tm"]); w["hour"] = w["tm"].dt.hour
    for c in ["ta", "hm", "ws"]:
        w[c] = pd.to_numeric(w[c], errors="coerce")
    w = w.dropna(subset=["ta", "hm", "ws"])
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    w[["tm", "stn_name", "hour", "ta", "hm", "ws"]].to_csv(DATA_PROCESSED / "weather_hourly.csv", index=False, encoding="utf-8-sig")
    print("기상", len(w), "행. 여름 시간대별 평균기온(수원):")
    print(w[w.stn_name.str.contains("수원")].groupby("hour")["ta"].mean().round(1).to_dict())

기상 4416 행. 여름 시간대별 평균기온(수원):
{0: 23.8, 1: 23.4, 2: 23.1, 3: 22.9, 4: 22.7, 5: 22.6, 6: 22.6, 7: 23.7, 8: 24.9, 9: 25.9, 10: 26.8, 11: 27.7, 12: 28.5, 13: 29.1, 14: 29.5, 15: 29.7, 16: 29.5, 17: 29.1, 18: 28.3, 19: 27.1, 20: 25.9, 21: 25.1, 22: 24.6, 23: 24.2}


In [3]:
if LOCAL_RUN:
    # 건물 → 격자별 건폐율·높이 (면적가중)
    grid = gpd.read_file(DATA_PROCESSED / "grid_features.geojson")
    grid = grid.drop(columns=["bld_cover", "bld_mean_h", "bld_max_h", "n_bld", "elev", "tpi"], errors="ignore")
    gL = grid.to_crs(CRS_LANDSAT)

    b = gpd.read_file(DATA_EXTERNAL / "buildings_3d" / "buildings.geojson").to_crs(CRS_LANDSAT)
    h = pd.to_numeric(b["height"], errors="coerce")
    gf = pd.to_numeric(b["grnd_flr"], errors="coerce")
    b["height"] = np.where(h.isna() | (h <= 0), gf * 3.3, h)   # height=0/null → 층수×3.3 보정
    b["height"] = pd.to_numeric(b["height"], errors="coerce").clip(lower=0).fillna(0)

    pairs = gpd.sjoin(gL[["grid_id", "geometry"]], b[["height", "geometry"]], predicate="intersects").reset_index(drop=True)
    gl = gL.set_index("grid_id").geometry.loc[pairs["grid_id"].values].reset_index(drop=True)
    bl = b.geometry.loc[pairs["index_right"].values].reset_index(drop=True)
    pairs["iarea"] = gl.intersection(bl, align=False).area
    pairs["hw"] = pairs["iarea"] * pairs["height"]
    agg = pairs.groupby("grid_id").agg(barea=("iarea", "sum"), hwsum=("hw", "sum"),
                                       bld_max_h=("height", "max"), n_bld=("height", "size"))
    agg["bld_cover"] = (agg["barea"] / 900).clip(0, 1)
    agg["bld_mean_h"] = agg["hwsum"] / agg["barea"]
    grid = grid.merge(agg[["bld_cover", "bld_mean_h", "bld_max_h", "n_bld"]], on="grid_id", how="left")
    for c in ["bld_cover", "bld_mean_h", "bld_max_h", "n_bld"]:
        grid[c] = grid[c].fillna(0)
    print(f"건물 있는 격자 {(grid.n_bld>0).sum():,}. 지구별 평균 건물높이(m):")
    print(grid[grid.n_bld>0].groupby("zoneName")["bld_mean_h"].mean().round(1).to_dict())

건물 있는 격자 25,054. 지구별 평균 건물높이(m):
{'고양창릉 공공주택지구': 1.1, '남양주왕숙 공공주택지구': 3.5, '성남분당': 27.1, '성남판교지구 택지개발사업': 23.6}


In [4]:
if LOCAL_RUN:
    # DEM → 격자 표고 + TPI(지형위치: 표고 − 반경 500m 이웃 평균)
    import rasterio, glob
    from rasterio.merge import merge
    from scipy.spatial import cKDTree

    srcs = [rasterio.open(t) for t in glob.glob(str(DATA_EXTERNAL / "dem" / "*.tif"))]
    mosaic, tr = merge(srcs)
    dem = mosaic[0]
    cent = grid.to_crs(CRS_SUBMIT).geometry.centroid   # 4326
    rows, cols = rasterio.transform.rowcol(tr, cent.x.values, cent.y.values)
    rows = np.clip(rows, 0, dem.shape[0]-1); cols = np.clip(cols, 0, dem.shape[1]-1)
    grid["elev"] = dem[rows, cols].astype(float)

    cxy = np.c_[grid.to_crs(CRS_LANDSAT).geometry.centroid.x, grid.to_crs(CRS_LANDSAT).geometry.centroid.y]
    tree = cKDTree(cxy)
    neigh = tree.query_ball_point(cxy, r=500)
    grid["tpi"] = [grid["elev"].iloc[i] - grid["elev"].iloc[n].mean() for i, n in enumerate(neigh)]
    print("표고/ TPI 지구별:")
    print(grid.groupby("zoneName")[["elev", "tpi"]].mean().round(1))

<프로젝트>\AppData\Local\Temp\ipykernel_14080\3131395602.py:10: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cent = grid.to_crs(CRS_SUBMIT).geometry.centroid   # 4326


표고/ TPI 지구별:
               elev  tpi
zoneName                
고양창릉 공공주택지구    26.3  0.4
남양주왕숙 공공주택지구   38.9  0.2
성남분당           59.5  0.4
성남판교지구 택지개발사업  71.8  1.0


In [5]:
if LOCAL_RUN:
    grid.to_crs(CRS_SUBMIT).to_file(DATA_PROCESSED / "grid_features.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

    import matplotlib.pyplot as plt
    from src.viz.fonts import setup_font
    setup_font()   # 한글 폰트 자동 적용(COMPAS 미보유 시 동봉 폰트 등록)
    fig, ax = plt.subplots(1, 3, figsize=(20, 7))
    grid.plot(column="bld_mean_h", cmap="viridis", ax=ax[0], linewidth=0, legend=True, legend_kwds={"shrink": 0.5}, vmax=30)
    ax[0].set_title("평균 건물높이 (m)")
    grid.plot(column="bld_cover", cmap="pink_r", ax=ax[1], linewidth=0, legend=True, legend_kwds={"shrink": 0.5})
    ax[1].set_title("건폐율")
    grid.plot(column="elev", cmap="terrain", ax=ax[2], linewidth=0, legend=True, legend_kwds={"shrink": 0.5})
    ax[2].set_title("표고 (m)")
    for a in ax: a.set_axis_off()
    plt.tight_layout(); plt.savefig(OUT_FIGURES / "fig08_external.png", dpi=120); plt.close()